In [ ]:
import time
from pathlib import WindowsPath

from epoch2_interface import Gen5Interface

In [ ]:
iterations = 1
xpt_path = WindowsPath(
    "C:/Users/Public/Documents/Plate Reader/DATA/Gen5 w WEI/Zhenzhen/pr2_all96wells_wUV.xpt"
)
print(xpt_path)
if not xpt_path.exists():
    print("WARNING: that experiment path doesn't exist!")

In [ ]:
platereader = Gen5Interface(com_port=4)
print(platereader.status)
if platereader.status != 0:
    print("There's a problem with the platereader! See error code above.")
    print("Try running the cleanup cell at the bottom of the notebook, then retry.")
else:
    print("Platereader OK")

In [ ]:
import smtplib

# Import the email modules we'll need
from email.message import EmailMessage


def send_email(platereader_status):
    """send an email with the platereader status"""
    # Open the plain text file whose name is in textfile for reading.
    text = "There was an error with the platereader! See error code above."
    msg = EmailMessage()
    msg.set_content(text)

    # me == the sender's email address
    # you == the recipient's email address
    msg["Subject"] = f"Platereader error: {platereader_status}"
    msg["From"] = "test"
    msg["To"] = "test"

    # Send the message via our own SMTP server.
    s = smtplib.SMTP("localhost")
    s.send_message(msg)
    s.quit()

In [ ]:
for i in range(iterations):
    if platereader.status == 0:
        print(f"Starting read {i + 1} of {iterations}")
        platereader.run_experiment(xpt_path)
        print(f"Completed read {i + 1} of {iterations}")
    else:
        print("There's a problem with the platereader! See error code above.")
        print("Try running the cleanup cell at the bottom of the notebook, then retry.")
        send_email(platereader.status)
        break

In [ ]:
if platereader is not None:
    del platereader
    platereader = None
    time.sleep(10)

In [ ]:
# Run this is there are communications issues with the Epoch2.
# Warning: this will stop ALL runs on ALL connected platereaders
import psutil

for proc in psutil.process_iter():
    # check whether the process name matches
    if proc.name() == "Gen5.exe":
        proc.kill()